# Lezione 42 — QLoRA: quantizzazione + LoRA

> **Modulo:** LoRA e adattamento efficiente · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 40 (LoRA da zero).
>
> Obiettivo pratico unico: capire la **quantizzazione** (mostrata davvero in
> NumPy: da float32 a interi e ritorno, con l'errore misurato) e come QLoRA la
> combini con LoRA per adattare modelli enormi su poca memoria.

## Teoria essenziale

Un modello grande occupa memoria soprattutto per i suoi **pesi** in float32
(4 byte ciascuno). La **quantizzazione** li memorizza con meno bit (es. int8, o
4-bit in QLoRA), dividendo per una *scala* e arrotondando a interi. Si risparmia
memoria al prezzo di un piccolo **errore** di arrotondamento.

**QLoRA** (Dettmers et al., 2023) mette insieme due idee:

1. **congela** il modello base **quantizzato a 4-bit** (poca memoria, non si
   addestra);
2. addestra **adapter LoRA** in precisione piu' alta *sopra* la base
   quantizzata.

Cosi' si adatta un modello enorme su una singola GPU: la base pesa poco (4-bit)
e si aggiorna solo un pugno di parametri LoRA.

In [1]:
import numpy as np

rng = np.random.default_rng(42)

def quantizza_int8(W):
    scala = np.abs(W).max() / 127.0
    q = np.round(W / scala).astype(np.int8)     # interi in [-127, 127]
    return q, scala

def dequantizza(q, scala):
    return q.astype(np.float32) * scala

W = rng.normal(size=(256, 256)).astype(np.float32)
q, scala = quantizza_int8(W)
W_ric = dequantizza(q, scala)

errore = np.linalg.norm(W - W_ric) / np.linalg.norm(W)
byte_fp32 = W.nbytes
byte_int8 = q.nbytes + 4                          # + la scala
print(f"memoria float32: {byte_fp32} byte")
print(f"memoria int8   : {byte_int8} byte  ({byte_fp32/byte_int8:.1f}x meno)")
print(f"errore relativo di quantizzazione: {errore:.4f}")

memoria float32: 262144 byte
memoria int8   : 65540 byte  (4.0x meno)
errore relativo di quantizzazione: 0.0113


Leggi i numeri: passare a int8 riduce la memoria ~4x con un errore relativo
piccolo. QLoRA spinge oltre (4-bit, ~8x) e recupera la qualita' con gli adapter
LoRA addestrabili sopra.

In [2]:
# la LoRA (in alta precisione) sopra una base quantizzata: l'output combina
# la base dequantizzata (congelata) e l'adapter addestrabile
d, k, r = 256, 256, 4
A = rng.normal(size=(r, k)).astype(np.float32) * 0.01
B = np.zeros((d, r), dtype=np.float32)

def forward_qlora(x, q_base, scala, B, A, alpha_su_r=1.0/r):
    base = dequantizza(q_base, scala)            # base 4/8-bit congelata
    return x @ base + alpha_su_r * (x @ B @ A)   # + adapter in alta precisione

x = rng.normal(size=(8, d)).astype(np.float32)
# all'avvio (B=0) l'output e' solo la base quantizzata
assert np.allclose(forward_qlora(x, q, scala, B, A), x @ dequantizza(q, scala))
print("QLoRA all'avvio = solo base quantizzata (adapter nullo)  ✓")

QLoRA all'avvio = solo base quantizzata (adapter nullo)  ✓


## Il progetto: Memory AI Lab, passo 42

Impacchetto una funzione che stima la memoria dei pesi con e senza
quantizzazione, per decidere se un modello "entra" nell'hardware disponibile.

In [3]:
def memoria_pesi(n_parametri, bit):
    byte = n_parametri * bit / 8
    return round(byte / 1e9, 3)   # in GB

for bit in [32, 16, 8, 4]:
    print(f"{bit:2d}-bit: {memoria_pesi(2e9, bit):6.3f} GB per 2 miliardi di parametri")

# controllo: 4-bit occupa 1/8 del float32
assert abs(memoria_pesi(2e9, 4) - memoria_pesi(2e9, 32) / 8) < 1e-6
print("verifica: 4-bit = 1/8 di float32  ✓")

32-bit:  8.000 GB per 2 miliardi di parametri
16-bit:  4.000 GB per 2 miliardi di parametri
 8-bit:  2.000 GB per 2 miliardi di parametri
 4-bit:  1.000 GB per 2 miliardi di parametri
verifica: 4-bit = 1/8 di float32  ✓


## Riepilogo (max 8 punti)

1. I pesi float32 costano 4 byte ciascuno: tanta memoria.
2. **Quantizzare** = memorizzarli con meno bit (int8, 4-bit) via scala +
   arrotondamento.
3. Si risparmia memoria al prezzo di un piccolo **errore**.
4. int8 ≈ 4x meno memoria; 4-bit ≈ 8x meno.
5. **QLoRA**: base congelata a 4-bit + adapter LoRA in alta precisione.
6. Cosi' modelli enormi si adattano su una sola GPU.
7. All'avvio (B=0) l'output e' solo la base quantizzata.
8. La quantizzazione decide se un modello "entra" nell'hardware.

## Quiz

1. Perche' la quantizzazione introduce un errore?
2. Cosa resta congelato e cosa si addestra in QLoRA?
3. Quanta memoria risparmia il 4-bit rispetto al float32?

*(Risposte: 1. si arrotondano i pesi a un insieme discreto di valori; 2. la base
quantizzata e' congelata, si addestrano solo gli adapter LoRA; 3. circa 8x, cioe'
1/8 della memoria.)*